In [1]:
import cv2
import subprocess
import time

def run_another_script():
    subprocess.run(["python", "testscript.py"])  # Replace "your_other_script.py" with your script name

# Load the pre-trained face cascade classifier
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

# Initialize the webcam
cap = cv2.VideoCapture(0)

face_detected = False

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Convert the frame to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces in the frame
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) > 0:
        if not face_detected:
            run_another_script()
            face_detected = True

    # If no faces detected, reset face_detected flag
    if len(faces) == 0:
        face_detected = False

    # Wait for the other script to complete before entering sleep mode
    if face_detected:
        time.sleep(5)  # Sleep for 5 seconds, adjust as needed

    cv2.imshow("Face Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # Press 'Esc' to exit the program
        break

cap.release()
cv2.destroyAllWindows()


Wake-up feature

In [1]:
import cv2
from wakeonlan import send_magic_packet
import ctypes

# Load the pre-trained face cascade classifier
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

# MAC address for WoL
target_mac_address = "BC-54-2F-42-62-38"

# Prevent the computer from entering sleep mode
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)  # Prevent system sleep

# Initialize the webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Convert the frame to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces in the frame
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Draw a green rectangle around each detected face
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow("Face Detection", frame)

    if len(faces) > 0:
        send_magic_packet(target_mac_address)  # Send WoL packet to the specified MAC address

    if cv2.waitKey(1) & 0xFF == 27:  # Press 'Esc' to exit the program
        break

# Release the prevention of sleep when you're done
ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)  # Reset sleep settings

cap.release()
cv2.destroyAllWindows()
